# Video Notebook 2: Quantum Entanglement — Animated

---

### Animations

1. **Bell State Creation** — Watch two qubits become entangled step-by-step
2. **Correlated Measurements** — Measure one qubit, see the other collapse instantly
3. **Entanglement Swapping** — Teleport entanglement between distant qubits
4. **CHSH Inequality Violation** — Animated proof that quantum beats classical
5. **Multi-Qubit Entanglement Growth** — GHZ state building up qubit by qubit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle
from IPython.display import HTML, Image, display
import os

os.makedirs('animations', exist_ok=True)
np.random.seed(42)
print('Ready to generate entanglement animations!')

## Animation 1: Bell State Creation

$$|\Phi^+\rangle = \frac{1}{\sqrt{2}}(|00\rangle + |11\rangle)$$

Created by: $H \otimes I$ followed by $\text{CNOT}$

In [ ]:
# Animation 1: Bell State Creation — Two Bloch Spheres

fig = plt.figure(figsize=(16, 7), facecolor='white')
ax1 = fig.add_subplot(131, projection='3d')
ax2 = fig.add_subplot(132, projection='3d')
ax_info = fig.add_subplot(133)

def draw_sphere(ax, title='', elev=20, azim=30):
    ax.clear()
    u = np.linspace(0, 2*np.pi, 30)
    v = np.linspace(0, np.pi, 15)
    xs = np.outer(np.cos(u), np.sin(v))
    ys = np.outer(np.sin(u), np.sin(v))
    zs = np.outer(np.ones_like(u), np.cos(v))
    ax.plot_wireframe(xs, ys, zs, color='#cccccc', alpha=0.12, linewidth=0.4)
    theta_eq = np.linspace(0, 2*np.pi, 80)
    ax.plot(np.cos(theta_eq), np.sin(theta_eq), 0, 'k-', alpha=0.1, linewidth=0.6)
    ax.text(0, 0, 1.35, '$|0\\rangle$', fontsize=12, ha='center', color='#2196F3', fontweight='bold')
    ax.text(0, 0, -1.35, '$|1\\rangle$', fontsize=12, ha='center', color='#F44336', fontweight='bold')
    ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_zlim(-1.5, 1.5)
    ax.set_axis_off(); ax.view_init(elev=elev, azim=azim)
    ax.set_title(title, fontsize=13, fontweight='bold', pad=5)

n_frames = 80
# Phases: 0-19 both at |0>, 20-39 H on qubit A, 40-59 CNOT, 60-79 entangled

def animate_bell(frame):
    ax_info.clear()
    
    if frame < 20:
        # Phase 1: Both qubits at |0>
        phase = 'Initial: $|00\\rangle$'
        z1, z2 = 1.0, 1.0
        x1, y1, x2, y2 = 0, 0, 0, 0
        c1, c2 = '#2196F3', '#2196F3'
        probs = [1, 0, 0, 0]
        step_text = 'Step 1: Initial state $|00\\rangle$'
    elif frame < 40:
        # Phase 2: Hadamard on qubit A
        t = (frame - 20) / 19
        t_smooth = 0.5 - 0.5 * np.cos(np.pi * t)
        phase = 'Applying H on Qubit A'
        z1 = 1.0 - t_smooth  # |0> to equator
        x1 = t_smooth  # Move to |+>
        y1, x2, y2 = 0, 0, 0
        z2 = 1.0
        c1, c2 = '#4CAF50', '#2196F3'
        probs = [0.5, 0, 0.5, 0] if t > 0.8 else [1-0.5*t_smooth, 0, 0.5*t_smooth, 0]
        step_text = 'Step 2: $H|0\\rangle = |+\\rangle$'
    elif frame < 60:
        # Phase 3: CNOT entangling
        t = (frame - 40) / 19
        t_smooth = 0.5 - 0.5 * np.cos(np.pi * t)
        phase = 'Applying CNOT'
        # Both shrink toward center (entangled = mixed single-qubit state)
        z1 = (1 - t_smooth) * 0
        x1 = (1 - t_smooth) * 1
        z2 = (1 - t_smooth) * 1
        y1, x2, y2 = 0, 0, 0
        c1 = '#9C27B0'
        c2 = '#9C27B0'
        probs = [0.5, 0, 0, 0.5]
        step_text = 'Step 3: CNOT creates entanglement!'
    else:
        # Phase 4: Entangled — pulsing
        pulse = 0.15 * np.sin((frame - 60) * 0.4)
        phase = 'Entangled: $|\\Phi^+\\rangle$'
        z1 = pulse
        x1 = pulse
        z2 = pulse
        y1, x2, y2 = 0, 0, 0
        c1 = '#9C27B0'
        c2 = '#9C27B0'
        probs = [0.5, 0, 0, 0.5]
        step_text = 'Bell State: $\\frac{1}{\\sqrt{2}}(|00\\rangle + |11\\rangle)$'
    
    # Draw spheres
    draw_sphere(ax1, 'Qubit A', azim=30 + frame*0.3)
    draw_sphere(ax2, 'Qubit B', azim=30 + frame*0.3)
    
    # State vectors
    norm1 = np.sqrt(x1**2 + y1**2 + z1**2)
    norm2 = np.sqrt(x2**2 + y2**2 + z2**2)
    if norm1 > 0.05:
        ax1.quiver(0,0,0, x1,y1,z1, color=c1, arrow_length_ratio=0.12, linewidth=3)
        ax1.scatter([x1],[y1],[z1], c=c1, s=100, edgecolor='black', linewidth=1.5, zorder=10)
    else:
        ax1.scatter([0],[0],[0], c=c1, s=150, edgecolor='black', linewidth=2, zorder=10, marker='*')
        ax1.text(0, 0, -0.3, 'Mixed', fontsize=9, ha='center', color=c1)
    
    if norm2 > 0.05:
        ax2.quiver(0,0,0, x2,y2,z2, color=c2, arrow_length_ratio=0.12, linewidth=3)
        ax2.scatter([x2],[y2],[z2], c=c2, s=100, edgecolor='black', linewidth=1.5, zorder=10)
    else:
        ax2.scatter([0],[0],[0], c=c2, s=150, edgecolor='black', linewidth=2, zorder=10, marker='*')
        ax2.text(0, 0, -0.3, 'Mixed', fontsize=9, ha='center', color=c2)
    
    # Entanglement link
    if frame >= 45:
        alpha = min(1, (frame - 45) / 10)
        for ax in [ax1, ax2]:
            ring_t = np.linspace(0, 2*np.pi, 40)
            ring_r = 0.3 + 0.1 * np.sin(frame * 0.3)
            ax.plot(ring_r * np.cos(ring_t), ring_r * np.sin(ring_t), 0,
                   color='#9C27B0', alpha=alpha*0.5, linewidth=2)
    
    # Info panel
    ax_info.set_xlim(0, 1); ax_info.set_ylim(0, 1); ax_info.axis('off')
    ax_info.text(0.5, 0.95, phase, fontsize=14, ha='center', va='top', fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8EAF6', alpha=0.9))
    ax_info.text(0.5, 0.82, step_text, fontsize=11, ha='center', va='top', style='italic')
    
    # Probability bars
    states = ['$|00\\rangle$', '$|01\\rangle$', '$|10\\rangle$', '$|11\\rangle$']
    colors_bar = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
    for i, (s, p, cb) in enumerate(zip(states, probs, colors_bar)):
        y_pos = 0.6 - i * 0.13
        ax_info.barh(y_pos, p * 0.8, height=0.08, left=0.1, color=cb, alpha=0.7, edgecolor='black')
        ax_info.text(0.05, y_pos, s, fontsize=11, ha='right', va='center')
        ax_info.text(0.1 + p * 0.8 + 0.02, y_pos, f'{p:.2f}', fontsize=10, va='center')
    ax_info.text(0.5, 0.72, 'State Probabilities', fontsize=11, ha='center', fontweight='bold')
    
    # Circuit diagram
    ax_info.plot([0.1, 0.9], [0.12, 0.12], 'k-', linewidth=2)
    ax_info.plot([0.1, 0.9], [0.04, 0.04], 'k-', linewidth=2)
    ax_info.text(0.05, 0.12, 'A', fontsize=10, ha='center', va='center', fontweight='bold')
    ax_info.text(0.05, 0.04, 'B', fontsize=10, ha='center', va='center', fontweight='bold')
    
    # H gate
    h_alpha = 0.3 if frame < 20 else (0.9 if frame < 40 else 0.5)
    h_color = '#4CAF50' if 20 <= frame < 40 else '#cccccc'
    ax_info.add_patch(FancyBboxPatch((0.25, 0.07), 0.1, 0.1, boxstyle='round,pad=0.02',
                     facecolor=h_color, edgecolor='black', alpha=h_alpha, linewidth=1.5))
    ax_info.text(0.3, 0.12, 'H', fontsize=12, ha='center', va='center', fontweight='bold')
    
    # CNOT
    cnot_alpha = 0.3 if frame < 40 else (0.9 if frame < 60 else 0.5)
    cnot_color = '#9C27B0' if 40 <= frame < 60 else '#cccccc'
    ax_info.scatter([0.55], [0.12], s=80, c=cnot_color, edgecolor='black', alpha=cnot_alpha, zorder=5)
    ax_info.plot([0.55, 0.55], [0.04, 0.12], color=cnot_color, linewidth=2, alpha=cnot_alpha)
    ax_info.scatter([0.55], [0.04], s=120, facecolors='none', edgecolors=cnot_color, linewidth=2, alpha=cnot_alpha, zorder=5)

anim1 = animation.FuncAnimation(fig, animate_bell, frames=n_frames, interval=80, blit=False)
anim1.save('animations/07_bell_state_creation.gif', writer='pillow', fps=12, dpi=100)
plt.close()
print('Saved: animations/07_bell_state_creation.gif')
display(Image(filename='animations/07_bell_state_creation.gif'))

## Animation 2: Entangled Measurement — Spooky Action

When we measure one qubit of an entangled pair, the other instantly collapses:  
Measure qubit A → get $|0\rangle$ → qubit B is **instantly** $|0\rangle$ (even light-years away!)

In [ ]:
# Animation 2: Spooky Action at a Distance

fig, ax = plt.subplots(figsize=(14, 7), facecolor='white')

n_frames_spooky = 80
outcome = 0  # Measure |0> on Alice's side

def animate_spooky(frame):
    ax.clear()
    ax.set_xlim(-0.5, 10.5); ax.set_ylim(-1, 6)
    ax.set_aspect('equal'); ax.axis('off')
    
    # Title
    ax.text(5, 5.5, 'Entangled Measurement: Spooky Action at a Distance', 
            fontsize=16, ha='center', fontweight='bold')
    
    # Alice and Bob
    alice_x, bob_x = 2, 8
    y_center = 2.5
    
    # Alice box
    ax.add_patch(FancyBboxPatch((alice_x-1, y_center-1), 2, 2, boxstyle='round,pad=0.1',
                 facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2))
    ax.text(alice_x, y_center+1.3, 'Alice', fontsize=14, ha='center', fontweight='bold', color='#1565C0')
    
    # Bob box
    ax.add_patch(FancyBboxPatch((bob_x-1, y_center-1), 2, 2, boxstyle='round,pad=0.1',
                 facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2))
    ax.text(bob_x, y_center+1.3, 'Bob', fontsize=14, ha='center', fontweight='bold', color='#E65100')
    
    # Distance label
    ax.annotate('', xy=(bob_x-1.2, y_center-1.5), xytext=(alice_x+1.2, y_center-1.5),
               arrowprops=dict(arrowstyle='<->', color='gray', lw=1.5))
    ax.text(5, y_center-1.8, 'Could be light-years apart!', fontsize=10, ha='center', 
            style='italic', color='gray')
    
    if frame < 20:
        # Phase 1: Entangled pair
        phase = 'Entangled: $|\\Phi^+\\rangle = \\frac{1}{\\sqrt{2}}(|00\\rangle + |11\\rangle)$'
        
        # Wavy entanglement line
        ex = np.linspace(alice_x+1, bob_x-1, 100)
        ey = y_center + 0.15 * np.sin(8*ex + frame*0.5)
        ax.plot(ex, ey, '-', color='#9C27B0', linewidth=2.5, alpha=0.7)
        
        # Qubit symbols (superposition - flickering)
        flicker = 0.5 + 0.5 * np.sin(frame * 0.6)
        ax.text(alice_x, y_center, '$|?\\rangle$', fontsize=28, ha='center', va='center',
               color='#9C27B0', alpha=flicker, fontweight='bold')
        ax.text(bob_x, y_center, '$|?\\rangle$', fontsize=28, ha='center', va='center',
               color='#9C27B0', alpha=flicker, fontweight='bold')
    
    elif frame < 35:
        # Phase 2: Alice measures
        phase = 'Alice MEASURES her qubit...'
        t = (frame - 20) / 14
        
        # Measurement flash
        flash_r = t * 1.5
        circle = plt.Circle((alice_x, y_center), flash_r, color='yellow', alpha=max(0, 0.8-t), zorder=3)
        ax.add_patch(circle)
        
        # Collapsing text
        if frame > 28:
            ax.text(alice_x, y_center, f'$|{outcome}\\rangle$', fontsize=30, ha='center', 
                   va='center', color='#1565C0', fontweight='bold')
        else:
            flicker = 0.5 + 0.5 * np.sin(frame * 2)
            ax.text(alice_x, y_center, '$|?\\rangle$', fontsize=28, ha='center', 
                   va='center', color='red', alpha=flicker, fontweight='bold')
        
        ax.text(bob_x, y_center, '$|?\\rangle$', fontsize=28, ha='center', va='center',
               color='#9C27B0', alpha=0.5 + 0.5*np.sin(frame*0.8), fontweight='bold')
        
        # Wavy line breaking
        ex = np.linspace(alice_x+1, bob_x-1, 100)
        ey = y_center + 0.15 * np.sin(8*ex + frame*0.5)
        ax.plot(ex, ey, '-', color='#9C27B0', linewidth=2.5, alpha=max(0, 1-t))
    
    elif frame < 55:
        # Phase 3: Correlation signal (instantaneous!)
        t = (frame - 35) / 19
        phase = 'INSTANT correlation! Bob\'s qubit collapses too!'
        
        ax.text(alice_x, y_center, f'$|{outcome}\\rangle$', fontsize=30, ha='center',
               va='center', color='#1565C0', fontweight='bold')
        
        # Shockwave from Alice to Bob
        wave_x = alice_x + 1 + t * (bob_x - alice_x - 2)
        if wave_x < bob_x - 1:
            ax.axvline(x=wave_x, color='#9C27B0', linewidth=3, alpha=0.6)
            ax.text(bob_x, y_center, '$|?\\rangle$', fontsize=28, ha='center', va='center',
                   color='#9C27B0', fontweight='bold')
        else:
            # Bob collapses
            flash_r = min(1.5, (frame - 35 - 12) * 0.3)
            if flash_r > 0:
                circle = plt.Circle((bob_x, y_center), flash_r, color='yellow', alpha=max(0, 0.6-flash_r/3), zorder=3)
                ax.add_patch(circle)
            ax.text(bob_x, y_center, f'$|{outcome}\\rangle$', fontsize=30, ha='center',
                   va='center', color='#E65100', fontweight='bold')
    
    else:
        # Phase 4: Both collapsed, showing correlation
        phase = f'Result: Both measured $|{outcome}\\rangle$ — Perfect correlation!'
        
        ax.text(alice_x, y_center, f'$|{outcome}\\rangle$', fontsize=30, ha='center',
               va='center', color='#1565C0', fontweight='bold')
        ax.text(bob_x, y_center, f'$|{outcome}\\rangle$', fontsize=30, ha='center',
               va='center', color='#E65100', fontweight='bold')
        
        # Correlation check mark
        pulse = 0.8 + 0.2 * np.sin((frame-55) * 0.3)
        ax.text(5, y_center, '= =', fontsize=30, ha='center', va='center',
               color='#4CAF50', fontweight='bold', alpha=pulse)
    
    ax.text(5, 0.3, phase, fontsize=13, ha='center', fontweight='bold',
           bbox=dict(boxstyle='round,pad=0.4', facecolor='lightyellow', edgecolor='gray', alpha=0.9))

anim2 = animation.FuncAnimation(fig, animate_spooky, frames=n_frames_spooky, interval=90, blit=False)
anim2.save('animations/08_spooky_action.gif', writer='pillow', fps=11, dpi=100)
plt.close()
print('Saved: animations/08_spooky_action.gif')
display(Image(filename='animations/08_spooky_action.gif'))

## Animation 3: CHSH Inequality — Quantum Beats Classical

Classical limit: $S \leq 2$. Quantum maximum: $S = 2\sqrt{2} \approx 2.83$.

The animation shows the CHSH value $S$ growing as we test more measurement angles, breaking the classical bound.

In [ ]:
# Animation 3: CHSH Inequality Violation

fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor='white')

n_frames_chsh = 70

# Precompute CHSH values for different angle settings
def chsh_value(theta):
    """Quantum CHSH value for measurement angle theta."""
    return 2 * np.sqrt(2) * abs(np.cos(theta))

angles_tested = []
chsh_values_list = []

def animate_chsh(frame):
    for ax in axes:
        ax.clear()
    
    # Left: Measurement angle dial
    ax = axes[0]
    theta = frame * np.pi / (4 * n_frames_chsh) * 4  # Sweep 0 to pi
    
    # Draw dial
    dial_angles = np.linspace(0, 2*np.pi, 100)
    ax.plot(np.cos(dial_angles), np.sin(dial_angles), 'k-', linewidth=2)
    ax.plot(np.cos(dial_angles)*0.95, np.sin(dial_angles)*0.95, '-', color='#eee', linewidth=8)
    
    # Alice's measurement direction
    ax.arrow(0, 0, 0.8*np.cos(0), 0.8*np.sin(0), head_width=0.06, head_length=0.04,
            fc='#2196F3', ec='#2196F3', linewidth=2)
    ax.text(0.95*np.cos(0), 0.95*np.sin(0), 'A', fontsize=12, color='#2196F3', 
           ha='center', fontweight='bold')
    
    # Bob's measurement direction (rotating)
    ax.arrow(0, 0, 0.8*np.cos(theta), 0.8*np.sin(theta), head_width=0.06, head_length=0.04,
            fc='#F44336', ec='#F44336', linewidth=2)
    ax.text(0.95*np.cos(theta), 0.95*np.sin(theta), 'B', fontsize=12, color='#F44336',
           ha='center', fontweight='bold')
    
    # Angle arc
    arc_t = np.linspace(0, theta, 30)
    ax.plot(0.4*np.cos(arc_t), 0.4*np.sin(arc_t), '-', color='#4CAF50', linewidth=2)
    ax.text(0.3*np.cos(theta/2), 0.3*np.sin(theta/2)+0.08, f'{np.degrees(theta):.0f}deg',
           fontsize=10, color='#4CAF50', ha='center')
    
    ax.set_xlim(-1.3, 1.3); ax.set_ylim(-1.3, 1.3)
    ax.set_aspect('equal'); ax.set_title('Measurement Angles', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.2)
    
    # Right: CHSH value building up
    ax2 = axes[1]
    
    # Current CHSH value
    s_val = chsh_value(np.pi/4)  # Optimal is pi/4
    # Simulate building up statistics
    progress = frame / n_frames_chsh
    current_s = 2 * np.sqrt(2) * progress * abs(np.cos(np.pi/4)) + (1-progress) * 1.5
    if frame > 40:
        current_s = 2 * np.sqrt(2) * abs(np.cos(np.pi/4)) + 0.1 * np.sin(frame * 0.3)
    
    angles_tested.append(np.degrees(theta))
    chsh_values_list.append(current_s)
    
    # Plot CHSH value over time
    ax2.plot(range(len(chsh_values_list)), chsh_values_list, 'b-', linewidth=2.5, label='Quantum $S$')
    ax2.axhline(y=2, color='red', linestyle='--', linewidth=2.5, label='Classical limit ($S=2$)')
    ax2.axhline(y=2*np.sqrt(2), color='green', linestyle=':', linewidth=2, label=f'Tsirelson bound ($2\\sqrt{{2}}$)')
    
    # Shade violation region
    ax2.fill_between(range(len(chsh_values_list)), 2, chsh_values_list, 
                     where=[v > 2 for v in chsh_values_list],
                     color='red', alpha=0.15, label='Violation!')
    
    ax2.set_xlim(0, n_frames_chsh); ax2.set_ylim(0, 3.2)
    ax2.set_xlabel('Experiment Number', fontsize=12)
    ax2.set_ylabel('CHSH Value $S$', fontsize=12)
    ax2.set_title(f'CHSH Test: S = {current_s:.3f}', fontsize=13, fontweight='bold',
                color='green' if current_s > 2 else 'black')
    ax2.legend(fontsize=9, loc='lower right')
    ax2.grid(True, alpha=0.3)
    
    if current_s > 2:
        ax2.text(len(chsh_values_list)//2, 2.9, 'CLASSICAL LIMIT BROKEN!', 
                fontsize=12, ha='center', color='red', fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.8))

anim3 = animation.FuncAnimation(fig, animate_chsh, frames=n_frames_chsh, interval=80, blit=False)
anim3.save('animations/09_chsh_violation.gif', writer='pillow', fps=12, dpi=100)
plt.close()
angles_tested.clear(); chsh_values_list.clear()
print('Saved: animations/09_chsh_violation.gif')
display(Image(filename='animations/09_chsh_violation.gif'))

## Animation 4: GHZ State — Multi-Qubit Entanglement Growth

Watch entanglement spread qubit-by-qubit as we build the GHZ state:

$$|\text{GHZ}_n\rangle = \frac{1}{\sqrt{2}}(|00\ldots0\rangle + |11\ldots1\rangle)$$

In [ ]:
# Animation 4: GHZ State Growing — Qubit by Qubit

fig, ax = plt.subplots(figsize=(14, 7), facecolor='white')

max_qubits = 6
n_frames_ghz = 90
frames_per_qubit = n_frames_ghz // max_qubits

def animate_ghz(frame):
    ax.clear()
    ax.set_xlim(-1, max_qubits + 1)
    ax.set_ylim(-2, 5)
    ax.set_aspect('equal')
    ax.axis('off')
    
    n_active = min(max_qubits, frame // frames_per_qubit + 1)
    local_t = (frame % frames_per_qubit) / frames_per_qubit
    
    ax.text(max_qubits/2, 4.5, f'Building GHZ State: {n_active} Qubits', 
            fontsize=16, ha='center', fontweight='bold')
    
    # State text
    zeros = '0' * n_active
    ones = '1' * n_active
    ax.text(max_qubits/2, 3.7, f'$|\\text{{GHZ}}_{n_active}\\rangle = \\frac{{1}}{{\\sqrt{{2}}}}(|{zeros}\\rangle + |{ones}\\rangle)$',
           fontsize=13, ha='center', style='italic')
    
    # Draw qubits
    for i in range(max_qubits):
        x = i + 0.5
        y = 2
        
        if i < n_active:
            # Active qubit
            if i == n_active - 1 and local_t < 0.7:
                # Animating this qubit joining
                alpha = local_t / 0.7
                size = 0.3 + 0.1 * alpha
                color = '#9C27B0'
            else:
                alpha = 1.0
                size = 0.4
                color = '#9C27B0'
            
            # Qubit circle
            circle = plt.Circle((x, y), size, facecolor=color, edgecolor='black',
                               linewidth=2, alpha=alpha, zorder=5)
            ax.add_patch(circle)
            ax.text(x, y, f'Q{i}', fontsize=10, ha='center', va='center',
                   color='white', fontweight='bold', alpha=alpha)
            
            # Entanglement connections
            if i > 0 and (i < n_active - 1 or local_t > 0.5):
                conn_alpha = alpha * 0.6
                # Wavy connection line
                cx = np.linspace(i - 0.5 + 0.4, x - 0.4, 30)
                cy = y + 0.08 * np.sin(10*cx + frame*0.3)
                ax.plot(cx, cy, '-', color='#9C27B0', linewidth=2, alpha=conn_alpha)
        else:
            # Inactive qubit
            circle = plt.Circle((x, y), 0.3, facecolor='#eeeeee', edgecolor='gray',
                               linewidth=1, alpha=0.4, zorder=3)
            ax.add_patch(circle)
            ax.text(x, y, f'Q{i}', fontsize=10, ha='center', va='center',
                   color='gray', alpha=0.4)
    
    # Entanglement strength meter
    meter_y = 0.0
    ax.text(0, meter_y + 0.5, 'Entanglement:', fontsize=11, fontweight='bold')
    strength = n_active / max_qubits
    if n_active == n_active and local_t < 0.7:
        strength = (n_active - 1 + local_t/0.7) / max_qubits
    
    bar_width = strength * (max_qubits - 0.5)
    gradient_colors = plt.cm.magma(np.linspace(0.2, 0.8, 10))
    for seg in range(int(bar_width * 10)):
        seg_x = 0.2 + seg * 0.1
        ax.add_patch(plt.Rectangle((seg_x, meter_y - 0.15), 0.09, 0.3,
                     facecolor=gradient_colors[min(seg, 9)], edgecolor='none'))
    ax.add_patch(plt.Rectangle((0.2, meter_y - 0.15), max_qubits - 0.5, 0.3,
                 facecolor='none', edgecolor='black', linewidth=1.5))
    ax.text(max_qubits - 0.1, meter_y, f'{strength*100:.0f}%', fontsize=11, 
           ha='left', va='center', fontweight='bold')
    
    # Bottom: circuit building up
    circuit_y = -1.2
    for i in range(max_qubits):
        x = i + 0.5
        ax.plot([0, max_qubits], [circuit_y - i*0.15, circuit_y - i*0.15], 
               'k-', linewidth=0.8, alpha=0.3)
    
    # H gate on first qubit
    ax.add_patch(FancyBboxPatch((0.3, circuit_y - 0.12), 0.4, 0.24, boxstyle='round,pad=0.02',
                 facecolor='#4CAF50', edgecolor='black', alpha=0.8))
    ax.text(0.5, circuit_y, 'H', fontsize=9, ha='center', va='center', color='white', fontweight='bold')
    
    # CNOT gates
    for i in range(1, n_active):
        cx = i + 0.1
        ax.scatter([cx], [circuit_y], s=30, c='black', zorder=5)
        ax.plot([cx, cx], [circuit_y, circuit_y - i*0.15], 'k-', linewidth=1.5)
        ax.scatter([cx], [circuit_y - i*0.15], s=50, facecolors='none', edgecolors='black', linewidth=1.5, zorder=5)

anim4 = animation.FuncAnimation(fig, animate_ghz, frames=n_frames_ghz, interval=80, blit=False)
anim4.save('animations/10_ghz_growth.gif', writer='pillow', fps=12, dpi=100)
plt.close()
print('Saved: animations/10_ghz_growth.gif')
display(Image(filename='animations/10_ghz_growth.gif'))

## Animation 5: Quantum Teleportation Protocol

Step-by-step visualization of quantum teleportation:  
Alice sends an unknown qubit state $|\psi\rangle$ to Bob using entanglement + 2 classical bits.

In [ ]:
# Animation 5: Quantum Teleportation Protocol

fig, ax = plt.subplots(figsize=(15, 8), facecolor='white')

n_frames_tp = 100
# Phases: 0-15 setup, 16-35 entangle, 36-55 alice measures, 56-75 classical bits, 76-99 bob corrects

def animate_teleport(frame):
    ax.clear()
    ax.set_xlim(-1, 16); ax.set_ylim(-1, 8)
    ax.axis('off')
    
    alice_x, bob_x = 3, 13
    mid_x = 8
    
    # Title
    ax.text(8, 7.5, 'Quantum Teleportation Protocol', fontsize=18, ha='center', fontweight='bold')
    
    # Alice region
    ax.add_patch(FancyBboxPatch((0.5, 0.5), 5, 5.5, boxstyle='round,pad=0.2',
                 facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2, alpha=0.3))
    ax.text(3, 6.3, 'ALICE', fontsize=14, ha='center', fontweight='bold', color='#1565C0')
    
    # Bob region
    ax.add_patch(FancyBboxPatch((10.5, 0.5), 5, 5.5, boxstyle='round,pad=0.2',
                 facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2, alpha=0.3))
    ax.text(13, 6.3, 'BOB', fontsize=14, ha='center', fontweight='bold', color='#E65100')
    
    # Step indicator
    steps = ['1. Prepare', '2. Entangle', '3. Bell Measure', '4. Send Bits', '5. Correct']
    step_idx = min(4, frame // 20)
    for i, step in enumerate(steps):
        color = '#4CAF50' if i < step_idx else ('#FF9800' if i == step_idx else 'gray')
        weight = 'bold' if i == step_idx else 'normal'
        ax.text(8, -0.3 - i*0.35, step, fontsize=9, ha='center', color=color, fontweight=weight)
    
    if frame < 16:
        # Phase 1: Setup - Alice has unknown qubit |psi>
        ax.text(3, 4.5, '$|\\psi\\rangle = \\alpha|0\\rangle + \\beta|1\\rangle$', 
               fontsize=14, ha='center', color='#9C27B0',
               bbox=dict(boxstyle='round', facecolor='#F3E5F5', alpha=0.8))
        ax.text(3, 3.5, 'Unknown state', fontsize=10, ha='center', style='italic', color='gray')
        pulse = 0.7 + 0.3 * np.sin(frame * 0.5)
        circle = plt.Circle((3, 4.5), 0.8, facecolor='#9C27B0', alpha=pulse*0.15, zorder=2)
        ax.add_patch(circle)
    
    elif frame < 36:
        # Phase 2: Create entangled pair
        t = (frame - 16) / 19
        ax.text(3, 4.5, '$|\\psi\\rangle$', fontsize=14, ha='center', color='#9C27B0')
        
        # Entangled pair emerging from center
        pair_spread = t * 5
        epr_alice_x = mid_x - pair_spread / 2
        epr_bob_x = mid_x + pair_spread / 2
        
        ax.scatter([epr_alice_x], [2.5], s=100, c='#9C27B0', edgecolor='black', zorder=5)
        ax.scatter([epr_bob_x], [2.5], s=100, c='#9C27B0', edgecolor='black', zorder=5)
        
        # Entanglement line
        ex = np.linspace(epr_alice_x, epr_bob_x, 50)
        ey = 2.5 + 0.1 * np.sin(10*ex + frame*0.5)
        ax.plot(ex, ey, '-', color='#9C27B0', linewidth=2, alpha=0.6)
        ax.text(mid_x, 3.2, 'EPR Pair', fontsize=11, ha='center', color='#9C27B0', fontweight='bold')
    
    elif frame < 56:
        # Phase 3: Alice does Bell measurement
        t = (frame - 36) / 19
        
        ax.scatter([3], [2.5], s=100, c='#9C27B0', edgecolor='black', zorder=5)
        ax.scatter([13], [2.5], s=100, c='#9C27B0', edgecolor='black', zorder=5)
        ax.text(3, 4.5, '$|\\psi\\rangle$', fontsize=14, ha='center', color='#9C27B0')
        
        # Measurement box
        m_alpha = min(1, t * 2)
        ax.add_patch(FancyBboxPatch((1.5, 2.8), 3, 2, boxstyle='round,pad=0.1',
                     facecolor='yellow', edgecolor='red', linewidth=2, alpha=m_alpha*0.5))
        ax.text(3, 3.8, 'Bell\nMeasurement', fontsize=12, ha='center', fontweight='bold',
               color='red', alpha=m_alpha)
        
        if t > 0.6:
            ax.text(3, 1.5, 'Result: 00', fontsize=12, ha='center', fontweight='bold',
                   color='#1565C0', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    elif frame < 76:
        # Phase 4: Send classical bits
        t = (frame - 56) / 19
        
        ax.text(3, 1.5, 'Result: 00', fontsize=12, ha='center', fontweight='bold', color='#1565C0')
        ax.scatter([13], [2.5], s=100, c='#9C27B0', edgecolor='black', zorder=5)
        
        # Classical bits flying to Bob
        bit_x = 3 + t * 10
        if bit_x < 13:
            ax.text(bit_x, 1.5, '00', fontsize=16, ha='center', fontweight='bold',
                   color='#1565C0', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
            ax.annotate('', xy=(bit_x + 0.5, 1.5), xytext=(bit_x - 0.1, 1.5),
                       arrowprops=dict(arrowstyle='->', color='#1565C0', lw=2))
        ax.text(8, 0.8, 'Classical Channel (2 bits)', fontsize=10, ha='center',
               style='italic', color='gray')
        ax.plot([3, 13], [1.1, 1.1], '--', color='gray', linewidth=1)
    
    else:
        # Phase 5: Bob applies correction -> has |psi>!
        t = (frame - 76) / 23
        
        ax.text(13, 1.5, 'Received: 00', fontsize=11, ha='center', color='#1565C0')
        
        if t < 0.5:
            ax.text(13, 3, 'Applying\ncorrection...', fontsize=12, ha='center', 
                   fontweight='bold', color='#E65100')
        else:
            pulse = 0.7 + 0.3 * np.sin(frame * 0.5)
            circle = plt.Circle((13, 4), 0.8, facecolor='#9C27B0', alpha=pulse*0.2, zorder=2)
            ax.add_patch(circle)
            ax.text(13, 4, '$|\\psi\\rangle$', fontsize=18, ha='center', color='#9C27B0',
                   fontweight='bold', bbox=dict(boxstyle='round', facecolor='#F3E5F5', alpha=0.8))
            ax.text(13, 2.8, 'TELEPORTED!', fontsize=14, ha='center', color='#4CAF50', fontweight='bold')

anim5 = animation.FuncAnimation(fig, animate_teleport, frames=n_frames_tp, interval=90, blit=False)
anim5.save('animations/11_teleportation.gif', writer='pillow', fps=11, dpi=100)
plt.close()
print('Saved: animations/11_teleportation.gif')
display(Image(filename='animations/11_teleportation.gif'))

## Summary

| File | Description |
|------|-------------|
| `07_bell_state_creation.gif` | Bell state built with H + CNOT, dual Bloch spheres |
| `08_spooky_action.gif` | Entangled measurement with instant correlation |
| `09_chsh_violation.gif` | CHSH inequality broken — quantum beats classical |
| `10_ghz_growth.gif` | GHZ entanglement spreading qubit by qubit |
| `11_teleportation.gif` | Full quantum teleportation protocol step-by-step |